# Day 48: Lab - Customer Churn Mini-Project

## Project Goal
Churn prediction is one of the most common real-world classification problems - subscription businesses (telecom, SaaS, streaming) want to identify customers likely to cancel so they can intervene (discounts, outreach) before they leave. This lab combines several ideas from the past few days - imbalanced classes, stratified splitting, tree-based models, and choosing the right evaluation metric - into one realistic mini-project. As with the housing lab, we use a synthetic dataset so the true relationship between features and churn is known and controllable. The cell below generates customer data (tenure, monthly charges, support calls) and derives a churn probability using a logistic function where more support calls and shorter tenure increase churn risk - a plausible, if simplified, business rule - then samples actual churn outcomes from that probability.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 500
df = pd.DataFrame({
    'tenure_months': np.random.randint(1, 60, n),
    'monthly_charges': np.round(np.random.uniform(20, 120, n), 2),
    'support_calls': np.random.poisson(2, n),
})
# logistic function converts a linear combination of features into a 0-1 probability;
# more support calls and shorter tenure push churn probability up
churn_prob = 1 / (1 + np.exp(-(-2 + df['support_calls'] * 0.5 - df['tenure_months'] * 0.03)))
df['churn'] = np.random.binomial(1, churn_prob)  # sample actual 0/1 churn outcome from that probability
print(df['churn'].value_counts())

## Step 1: Train-Test Split
As with any classification project, we hold out a test set the model never trains on, so we get an honest estimate of how well it generalizes. Churn datasets are typically imbalanced (most customers don't churn in a given period), so `stratify=y` is important here - it forces both the train and test sets to preserve the same proportion of churners as the full dataset. Without stratification, a random split could by chance leave too few (or too many) churners in the test set, making evaluation metrics noisy and unreliable. `X` holds the customer features and `y` holds the churn label (0 = stayed, 1 = churned) we're trying to predict.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['churn'])  # features: tenure, monthly charges, support calls
y = df['churn']  # target: did the customer churn (1) or not (0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)  # stratify preserves churn ratio in both splits

## Step 2: Train a Classifier
`RandomForestClassifier` is a strong default choice for tabular business data like this: it's an ensemble of many decision trees, handles non-linear relationships and feature interactions automatically, needs little feature scaling or preprocessing, and tends to be robust without heavy tuning. `n_estimators=100` sets the number of trees in the forest - more trees generally give more stable predictions at the cost of more computation, with diminishing returns past a point. `class_weight='balanced'` addresses the class imbalance we saw in Day 46 - it automatically upweights the minority (churn) class during training so the model doesn't just default to predicting "no churn" for everyone. This cell only fits the model; results are checked in the next step.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)  # 100 trees, upweighted churn class
model.fit(X_train, y_train)

## Step 3: Evaluate with the Right Metrics
In a churn business context, the cost of missing an actual churner (a false negative - predicting "will stay" when they actually leave) is usually much higher than the cost of a false alarm (flagging a loyal customer as at-risk, which just wastes a bit of retention-team effort). This means recall on the churn class - the fraction of actual churners the model successfully identifies - is often the metric that matters most, even more than overall accuracy or precision. `classification_report` conveniently prints precision, recall, and F1-score broken down per class (0 = stayed, 1 = churned), plus overall accuracy, so you can check the churn-class (label 1) recall specifically rather than relying on a single blended number. When reading the output, look at the row for class 1: a low recall there means the model is letting a lot of actual churners slip through undetected, which is the costly mistake in this business scenario.

In [ ]:
from sklearn.metrics import classification_report

preds = model.predict(X_test)
print(classification_report(y_test, preds))  # check recall for class 1 (churn) specifically - that's the costly miss to avoid